# Import

In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report
import lightgbm as lgb
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Add src/ to path (once, so imports work)
sys.path.append(str(Path().resolve().parent / "src"))
# 
# Enable autoreload for Jupyter notebooks
%load_ext autoreload
%autoreload 2

from paths import DATA_DATASETS
from helper_functions import get_master_dataframe
import feature_construction as fc

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
# Get master dataframe - constructed of transactions, outfits, and outfit_clusters
master = get_master_dataframe()


test_customers = pd.read_csv(DATA_DATASETS / "test_customers.csv", sep=";")
baseline_pred = pd.read_csv(DATA_DATASETS / "pred.csv")

Master shape: (60897, 11)
   customer.id                                outfit.id rentalPeriod.start  \
0         3448  outfit.5c081909537b42239e465d2d615c705f         2023-03-26   
1         2924  outfit.c34969dd8b334064aa90bfb60c8ec308         2023-03-27   

  rentalPeriod.end  cluster         cluster_name  pricePerWeek  pricePerMonth  \
0       2023-04-25      3.0  Sweaters & Knitwear         750.0         1500.0   
1       2023-04-26      7.0      Jackets & Coats         990.0         1980.0   

   retailPrice  duration_days  revenue  
0       2500.0             30   1500.0  
1       3900.0             30   1980.0  


In [46]:
# Fold structure
# TIMEFRAME_START  = pd.Timestamp("2017-04-01")
TRAIN_CUTOFF     = pd.Timestamp("2021-09-06")  # features end here for training
LABEL_END        = pd.Timestamp("2022-09-06")  # labels end here for training
FINAL_CUTOFF     = pd.Timestamp("2023-09-06")  # features end here for submission

X_train, X_test, y_train, y_test, id_train, id_test = fc.generate_train_test_splits(df=master, train_cutoff=TRAIN_CUTOFF, label_end=LABEL_END, final_cutoff=FINAL_CUTOFF)

In [ ]:
# Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Baseline Model
sfs_selected_features = ['recency', 'monetary', 'avg_revenue', 'weighted_rev', 'monetary_x_frequency', 'tenure_days', 'revenue_per_day', 'rentals_per_day', 'n_summer', 'active_months', 'revenue_trend', 'pct_weekly_rentals', 'cluster_affinity_0', 'cluster_affinity_1', 'cluster_affinity_2', 'cluster_affinity_7', 'max_retail_price', 'avg_price_per_week']

X_train_sfs = X_train[sfs_selected_features]
X_test_sfs = X_test[sfs_selected_features]

# 2-stage model
# Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Churn Model
sfs_selected_features_2s_clf = [
    'recency',
    'monetary_x_frequency',
    'pct_weekly_rentals',
    'cluster_affinity_6'
]

X_train_sfs_clf = X_train[sfs_selected_features_2s_clf]
X_test_sfs_clf = X_test[sfs_selected_features_2s_clf]

# Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Regression Model
# sfs_selected_features_2s_reg = ['recency', 'monetary', 'tenure_days', 'avg_days_between_rentals', 'revenue_trend']
sfs_selected_features_2s_reg = [
    'recency', 
    'monetary', 
    'avg_revenue', 
    'weighted_rev', 
    'monetary_x_frequency', 
    'tenure_days', 
    'revenue_per_day', 
    'rentals_per_day', 
    'revenue_trend', 
    'max_retail_price', 
    'avg_price_per_week'
]

X_train_sfs_reg = X_train[sfs_selected_features_2s_reg]
X_test_sfs_reg = X_test[sfs_selected_features_2s_reg]

In [48]:
# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.3
)

# Train fold
lgbm_model.fit(X_train_sfs, y_train)

train_predictions = np.clip(np.array(lgbm_model.predict(X_train_sfs)), 0, None)
train_mae = mean_absolute_error(y_train, train_predictions)
print(f"MAE on training fold: {train_mae:.2f} NOK")


# Test fold
test_predictions = np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)

test_mae = mean_absolute_error(y_test, test_predictions)
print(f"MAE on test fold: {test_mae:.2f} NOK")

MAE on training fold: 377.46 NOK
MAE on test fold: 2499.41 NOK


## Two-stage Model

In [49]:
# Stage 1: Who is going to be active?
y_train_churn = (y_train > 0).astype(int).values.ravel()
y_test_churn  = (y_test  > 0).astype(int).values.ravel()

n_inactive = (y_train_churn == 0).sum()
n_active   = (y_train_churn == 1).sum()
print(f"Training: {n_active} aktiv, {n_inactive} inaktiv ({n_inactive/n_active:.1f}:1)")

churn_model = lgb.LGBMClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    # scale_pos_weight  = n_inactive / n_active,
    random_state      = 42,
    verbose           = -1
)
churn_model.fit(X_train_sfs_clf, y_train_churn)

churn_prob_test  = churn_model.predict_proba(X_test_sfs_clf)[:, 1]
churn_pred_test  = churn_model.predict(X_test_sfs_clf)

print("Churn Classifier — Test Fold:")
print(classification_report(y_test_churn, churn_pred_test, target_names=["Inactive", "Active"]))



# Stage 2: How much revenue do active customers generate?
active_mask = (y_train > 0).values.ravel()

print(f"\nRevenue Model trained on {active_mask.sum()} active customers.")

# Log-transform the target for better modeling (optional, but often helps with skewed revenue data)
# y_train_log = np.log1p(y_train.values.ravel()[active_mask])

revenue_model = lgb.LGBMRegressor(
    n_estimators      = 600,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 0,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.8
)

# revenue_model.fit(X_train[active_mask], y_train_log)
revenue_model.fit(X_train_sfs_reg[active_mask], y_train.values.ravel()[active_mask])
# revenue_model.fit(X_train_sfs_reg[active_mask], y_train_log)

# Predict and transform back from log scale
pred_log = np.array(revenue_model.predict(X_test_sfs_reg))
revenue_if_active = np.clip(pred_log, 0, None)
#revenue_if_active = np.clip(np.expm1(pred_log), 0, None)

# Hard: entweder 0 oder predicted revenue
final_pred_hard = np.where(churn_pred_test == 1, revenue_if_active, 0)

# Soft: P(aktiv) × expected revenue — oft besser für MAE
final_pred_soft = churn_prob_test * revenue_if_active

mae_hard      = mean_absolute_error(y_test, final_pred_hard)
mae_soft      = mean_absolute_error(y_test, final_pred_soft)
mae_baseline  = mean_absolute_error(y_test, np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None))

print(f"\nResults on Test Fold:")
print(f"Single-Stage LightGBM: {mae_baseline:.2f} NOK")
print(f"Two-Stage Hard: {mae_hard:.2f} NOK")
print(f"Two-Stage Soft: {mae_soft:.2f} NOK")

Training: 439 aktiv, 5504 inaktiv (12.5:1)
Churn Classifier — Test Fold:
              precision    recall  f1-score   support

    Inactive       0.98      0.98      0.98      6193
      Active       0.76      0.76      0.76       616

    accuracy                           0.96      6809
   macro avg       0.87      0.87      0.87      6809
weighted avg       0.96      0.96      0.96      6809


Revenue Model trained on 439 active customers.

Results on Test Fold:
Single-Stage LightGBM: 2499.41 NOK
Two-Stage Hard: 2596.65 NOK
Two-Stage Soft: 2757.01 NOK


In [55]:
y_test_flat = y_test.values.ravel()

# Diagnostics
results = pd.DataFrame({
    "actual":         y_test_flat,
    "pred_single":    np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None),
    "pred_hard":      final_pred_hard,
    "pred_soft":      final_pred_soft,
    "churn_prob":     churn_prob_test,
    "error_single":   np.abs(y_test_flat - np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)),
    "error_hard":     np.abs(y_test_flat - final_pred_hard),
    "error_soft":     np.abs(y_test_flat - final_pred_soft),
})

results["bucket"] = pd.cut(results["actual"],
    bins=[-0.01, 0.01, 1000, 5000, 15000, 30000, 45000, 60000, 999999],
    labels=["Churned (0)", "1–1k NOK", "1k–5k NOK", "5k-15k NOK", "15k-30k NOK", "30k-45k NOK", "45k-60k NOK", "60k+ NOK"])

print("\nMAE for each customer segment:")
print(results.groupby("bucket", observed=True).agg(
    n           = ("actual",      "count"),
    mae_single  = ("error_single","mean"),
    mae_hard    = ("error_hard",  "mean"),
    mae_soft    = ("error_soft",  "mean"),
    avg_actual  = ("actual",      "mean"),
).round(2).to_string())

# Threshold Optimisation
# Find optimal threshold for churn classifier to minimize MAE
print("\nThreshold Optimization:")
thresholds = np.arange(0.5, 1, 0.01)
threshold_results = []

for t in thresholds:
    pred = np.where(churn_prob_test >= t, revenue_if_active, 0)
    mae  = mean_absolute_error(y_test, pred)
    threshold_results.append({"threshold": t, "mae": mae})

thresh_df = pd.DataFrame(threshold_results)
best_thresh = thresh_df.loc[thresh_df["mae"].idxmin(), "threshold"]
best_mae    = thresh_df["mae"].min()

print(thresh_df.to_string(index=False))
print(f"\nBest Threshold: {best_thresh:.2f} → MAE: {best_mae:.2f} NOK")


MAE for each customer segment:
                n  mae_single  mae_hard  mae_soft  avg_actual
bucket                                                       
Churned (0)  6193      409.15    426.15    629.19        0.00
1k–5k NOK      84     9271.41  12799.49  11998.82     2951.35
5k-15k NOK    122    13290.47  13622.36  12723.58     9855.13
15k-30k NOK   103    15932.48  17481.83  17218.30    21818.84
30k-45k NOK   102    21528.81  18758.76  18857.29    38050.78
45k-60k NOK    96    29218.38  27934.71  28080.11    51650.07
60k+ NOK      109    49930.59  54207.66  54340.42    96566.65

Threshold Optimization:
 threshold         mae
      0.50 2596.653132
      0.51 2586.592712
      0.52 2584.914204
      0.53 2584.914204
      0.54 2584.816781
      0.55 2582.518775
      0.56 2581.349905
      0.57 2583.016626
      0.58 2583.639365
      0.59 2576.412088
      0.60 2576.412088
      0.61 2583.375698
      0.62 2581.339539
      0.63 2577.415910
      0.64 2573.837434
      0.65 2570.8